In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
# ─── CONFIG ───────────────────────────────────────────────────────────────────
CONFIG = {
    "image_size"        : 128,
    "in_channels"       : 1,
    "num_classes"       : 10,
    "kernel_sizes"      : [1, 2, 3, 4],
    "filters_per_kernel": 3,
    "conv2d_layers"     : [
        # (out_ch, kernel_h, kernel_w, stride_h, stride_w)
        (32,  2, 3, 1, 1),   # collapses height 2→1, keeps width
        (64,  1, 3, 1, 2),   # reduces width 128→64
        (128, 1, 3, 1, 2),   # reduces width 64→32
        (64,  1, 3, 1, 2),   # reduces width 32→16
    ],
    "fc_layers"         : [256, 128],
    "dropout"           : 0.3,
    "batch_size"        : 64,
    "epochs"            : 15,
    "lr"                : 1e-3,
    "data_dir"          : "./kmnist_data",
}
# shape flow:
# (B, 16, 2, 128)
# → (B, 32, 1, 128) → (B, 64, 1, 64) → (B, 128, 1, 32) → (B, 64, 1, 16)
# → GlobalAvgPool → (B, 64)
# → FC: 64→256→128→10
# ──────────────────────────────────────────────────────────────────────────────


# ─── DATA ─────────────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.1918,), (0.3483,)),
])

train_ds = datasets.KMNIST(CONFIG["data_dir"], train=True,  download=True, transform=transform)
val_ds   = datasets.KMNIST(CONFIG["data_dir"], train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device  : {device}")
print(f"Train samples : {len(train_ds)}")
print(f"Val samples   : {len(val_ds)}")

imgs, labels = next(iter(train_loader))
print(f"Image shape   : {imgs.shape}")
# ──────────────────────────────────────────────────────────────────────────────


Using device  : cuda
Train samples : 60000
Val samples   : 10000
Image shape   : torch.Size([64, 1, 128, 128])


In [2]:




# ─── ROW EXTRACTOR ────────────────────────────────────────────────────────────
class RowExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size   = cfg["image_size"]
        n_filt     = cfg["filters_per_kernel"]
        in_ch      = cfg["in_channels"]
        self.convs = nn.ModuleList([
            nn.Conv2d(in_ch, n_filt, kernel_size=(k, img_size), stride=1, padding=0)
            for k in cfg["kernel_sizes"]
        ])
        self.img_size = img_size
        self.n_filt   = n_filt

    def forward(self, x):
        # x: (B, 1, H, W)
        outs = []
        for conv in self.convs:
            o = conv(x)              # (B, n_filt, H-k+1, 1)
            o = o.squeeze(-1)        # (B, n_filt, H-k+1)
            # pool to fixed length img_size
            o = F.adaptive_avg_pool1d(o, self.img_size)  # (B, n_filt, img_size)
            # mean over filters → single channel per pair
            o = o.mean(dim=1, keepdim=True)              # (B, 1, img_size)
            outs.append(o)
        return outs   # 4 x (B, 1, 128)


# ─── COLUMN EXTRACTOR ─────────────────────────────────────────────────────────
class ColExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size   = cfg["image_size"]
        n_filt     = cfg["filters_per_kernel"]
        in_ch      = cfg["in_channels"]
        self.convs = nn.ModuleList([
            nn.Conv2d(in_ch, n_filt, kernel_size=(img_size, k), stride=1, padding=0)
            for k in cfg["kernel_sizes"]
        ])
        self.img_size = img_size
        self.n_filt   = n_filt

    def forward(self, x):
        outs = []
        for conv in self.convs:
            o = conv(x)              # (B, n_filt, 1, W-k+1)
            o = o.squeeze(2)         # (B, n_filt, W-k+1)
            o = F.adaptive_avg_pool1d(o, self.img_size)  # (B, n_filt, img_size)
            o = o.mean(dim=1, keepdim=True)              # (B, 1, img_size)
            outs.append(o)
        return outs   # 4 x (B, 1, 128)


# ─── PRCCN v2 — 2×128 PATCH MODEL ────────────────────────────────────────────
class PRCCNv2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.row_extractor = RowExtractor(cfg)
        self.col_extractor = ColExtractor(cfg)

        n_pairs  = len(cfg["kernel_sizes"]) ** 2   # 16
        img_size = cfg["image_size"]               # 128

        # ── Conv2d blocks on (B, 16, 2, 128) ─────────────────────────────
        conv2d_blocks = []
        in_ch = n_pairs   # 16
        for idx, (out_ch, kh, kw, sh, sw) in enumerate(cfg["conv2d_layers"]):
            pad_h = 0          # no height padding — we want height to collapse
            pad_w = kw // 2    # same padding along width
            conv2d_blocks += [
                nn.Conv2d(in_ch, out_ch,
                          kernel_size=(kh, kw),
                          stride=(sh, sw),
                          padding=(pad_h, pad_w)),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
            ]
            in_ch = out_ch
        self.conv2d_blocks = nn.Sequential(*conv2d_blocks)

        # GlobalAvgPool → (B, last_ch, 1, 1) → flatten → (B, last_ch)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        flat_size        = in_ch   # 64

        # ── FC layers ─────────────────────────────────────────────────────
        fc_blocks = []
        prev = flat_size
        for hidden in cfg["fc_layers"]:
            fc_blocks += [
                nn.Linear(prev, hidden),
                nn.ReLU(),
                nn.Dropout(cfg["dropout"]),
            ]
            prev = hidden
        fc_blocks.append(nn.Linear(prev, cfg["num_classes"]))
        self.fc = nn.Sequential(*fc_blocks)

        # ── parameter breakdown ───────────────────────────────────────────
        total       = sum(p.numel() for p in self.parameters())
        row_p       = sum(p.numel() for p in self.row_extractor.parameters())
        col_p       = sum(p.numel() for p in self.col_extractor.parameters())
        conv2d_p    = sum(p.numel() for p in self.conv2d_blocks.parameters())
        fc_p        = sum(p.numel() for p in self.fc.parameters())
        print(f"PRCCNv2 ready")
        print(f"  n_pairs       : {n_pairs}")
        print(f"  patch shape   : (B, {n_pairs}, 2, {img_size})")
        print(f"  flat_size     : {flat_size}")
        print(f"  total params  : {total:,}")
        print(f"  -- row 2D     : {row_p:,}")
        print(f"  -- col 2D     : {col_p:,}")
        print(f"  -- conv2d     : {conv2d_p:,}")
        print(f"  -- fc         : {fc_p:,}")

    def forward(self, x):
        row_vecs = self.row_extractor(x)   # 4 x (B, 1, 128)
        col_vecs = self.col_extractor(x)   # 4 x (B, 1, 128)

        # build 16 patches each (B, 1, 2, 128)
        patches = []
        for r in row_vecs:
            for c in col_vecs:
                patch = torch.stack([r, c], dim=2)   # (B, 1, 2, 128)
                patches.append(patch)

        # stack 16 patches as channels → (B, 16, 2, 128)
        out = torch.cat(patches, dim=1)              # (B, 16, 2, 128)

        out = self.conv2d_blocks(out)                # (B, 64, 1, 16)
        out = self.global_pool(out)                  # (B, 64, 1, 1)
        out = out.flatten(1)                         # (B, 64)
        return self.fc(out)                          # (B, num_classes)


# ─── TRAIN EPOCH ──────────────────────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze().long().to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


# ─── VALIDATE ─────────────────────────────────────────────────────────────────
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.squeeze().long().to(device)
            out    = model(imgs)
            loss   = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            all_probs.append(torch.softmax(out, dim=1).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    return total_loss / total, correct / total, auc


# ─── TRAIN ────────────────────────────────────────────────────────────────────
model     = PRCCNv2(CONFIG).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val_auc = 0.0

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc      = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d}/{CONFIG['epochs']}  "
          f"train_loss: {train_loss:.4f}  train_acc: {train_acc:.4f}  "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}  val_auc: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), "prccnv2_kmnist_best.pth")
        print(f"  >> best model saved (val_auc: {val_auc:.4f})")

print(f"\nTraining complete. Best val_auc: {best_val_auc:.4f}")

PRCCNv2 ready
  n_pairs       : 16
  patch shape   : (B, 16, 2, 128)
  flat_size     : 64
  total params  : 117,762
  -- row 2D     : 3,852
  -- col 2D     : 3,852
  -- conv2d     : 59,232
  -- fc         : 50,826
Epoch 01/15  train_loss: 0.6933  train_acc: 0.7742  val_loss: 0.7461  val_acc: 0.7561  val_auc: 0.9699
  >> best model saved (val_auc: 0.9699)
Epoch 02/15  train_loss: 0.3518  train_acc: 0.8903  val_loss: 0.5346  val_acc: 0.8279  val_auc: 0.9834
  >> best model saved (val_auc: 0.9834)
Epoch 03/15  train_loss: 0.2777  train_acc: 0.9130  val_loss: 0.4610  val_acc: 0.8504  val_auc: 0.9877
  >> best model saved (val_auc: 0.9877)
Epoch 04/15  train_loss: 0.2314  train_acc: 0.9288  val_loss: 0.4520  val_acc: 0.8539  val_auc: 0.9883
  >> best model saved (val_auc: 0.9883)
Epoch 05/15  train_loss: 0.2078  train_acc: 0.9364  val_loss: 0.4064  val_acc: 0.8731  val_auc: 0.9907
  >> best model saved (val_auc: 0.9907)
Epoch 06/15  train_loss: 0.1810  train_acc: 0.9433  val_loss: 0.3987  v

In [ ]:


# ─── CELL 2 — PRCCNv2 MODEL AND TRAINING ─────────────────────────────────────

# ─── ROW EXTRACTOR ────────────────────────────────────────────────────────────
class RowExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size    = cfg["image_size"]
        n_kernels   = 16                  # 16 direct row kernels
        in_ch       = cfg["in_channels"]

        # 16 kernels each of size 1×img_size, slides top to bottom
        self.conv = nn.Conv2d(in_ch, n_kernels,
                              kernel_size=(1, img_size),
                              stride=1, padding=0)
        self.img_size  = img_size
        self.n_kernels = n_kernels

    def forward(self, x):
        # x: (B, 1, H, W)
        o = self.conv(x)                          # (B, 16, H, 1)
        o = o.squeeze(-1)                         # (B, 16, H)
        o = F.adaptive_avg_pool1d(o, self.img_size)  # (B, 16, 128)
        return o   # (B, 16, 128)


# ─── COLUMN EXTRACTOR ─────────────────────────────────────────────────────────
class ColExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size    = cfg["image_size"]
        n_kernels   = 16
        in_ch       = cfg["in_channels"]

        # 16 kernels each of size img_size×1, slides left to right
        self.conv = nn.Conv2d(in_ch, n_kernels,
                              kernel_size=(img_size, 1),
                              stride=1, padding=0)
        self.img_size  = img_size
        self.n_kernels = n_kernels

    def forward(self, x):
        # x: (B, 1, H, W)
        o = self.conv(x)                          # (B, 16, 1, W)
        o = o.squeeze(2)                          # (B, 16, W)
        o = F.adaptive_avg_pool1d(o, self.img_size)  # (B, 16, 128)
        return o   # (B, 16, 128)


# ─── PRCCNv2 ──────────────────────────────────────────────────────────────────
class PRCCNv2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.row_extractor = RowExtractor(cfg)
        self.col_extractor = ColExtractor(cfg)

        n_pairs  = 16
        img_size = cfg["image_size"]

        # ── Conv2d blocks on (B, 16, 2, 128) ─────────────────────────────
        conv2d_blocks = []
        in_ch = n_pairs   # 16
        for (out_ch, kh, kw, sh, sw) in cfg["conv2d_layers"]:
            pad_h = 0        # no height padding — height collapses in first layer
            pad_w = kw // 2  # same padding along width
            conv2d_blocks += [
                nn.Conv2d(in_ch, out_ch,
                          kernel_size=(kh, kw),
                          stride=(sh, sw),
                          padding=(pad_h, pad_w)),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
            ]
            in_ch = out_ch
        self.conv2d_blocks = nn.Sequential(*conv2d_blocks)

        # GlobalAvgPool → (B, last_ch, 1, 1) → flatten → (B, last_ch)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        flat_size        = in_ch   # 64

        # ── FC layers ─────────────────────────────────────────────────────
        fc_blocks = []
        prev = flat_size
        for hidden in cfg["fc_layers"]:
            fc_blocks += [
                nn.Linear(prev, hidden),
                nn.ReLU(),
                nn.Dropout(cfg["dropout"]),
            ]
            prev = hidden
        fc_blocks.append(nn.Linear(prev, cfg["num_classes"]))
        self.fc = nn.Sequential(*fc_blocks)

        # ── parameter breakdown ───────────────────────────────────────────
        total      = sum(p.numel() for p in self.parameters())
        row_p      = sum(p.numel() for p in self.row_extractor.parameters())
        col_p      = sum(p.numel() for p in self.col_extractor.parameters())
        conv2d_p   = sum(p.numel() for p in self.conv2d_blocks.parameters())
        fc_p       = sum(p.numel() for p in self.fc.parameters())
        print(f"PRCCNv2 ready")
        print(f"  patch tensor  : (B, 16, 2, {img_size})")
        print(f"  flat_size     : {flat_size}")
        print(f"  total params  : {total:,}")
        print(f"  -- row kernels: {row_p:,}")
        print(f"  -- col kernels: {col_p:,}")
        print(f"  -- conv2d     : {conv2d_p:,}")
        print(f"  -- fc         : {fc_p:,}")

    def forward(self, x):
        row_out = self.row_extractor(x)   # (B, 16, 128)
        col_out = self.col_extractor(x)   # (B, 16, 128)

        # pair channel i with channel i — stack along height → (B, 16, 2, 128)
        # row_out[:, i, :] is shape (B, 128) — unsqueeze to (B, 1, 128)
        # stack row and col for each channel along dim=2
        row_out = row_out.unsqueeze(2)    # (B, 16, 1, 128)
        col_out = col_out.unsqueeze(2)    # (B, 16, 1, 128)
        out     = torch.cat([row_out, col_out], dim=2)  # (B, 16, 2, 128)

        out = self.conv2d_blocks(out)     # (B, 64, 1, 16)
        out = self.global_pool(out)       # (B, 64, 1, 1)
        out = out.flatten(1)              # (B, 64)
        return self.fc(out)               # (B, num_classes)


# ─── TRAIN EPOCH ──────────────────────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze().long().to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


# ─── VALIDATE ─────────────────────────────────────────────────────────────────
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.squeeze().long().to(device)
            out    = model(imgs)
            loss   = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            all_probs.append(torch.softmax(out, dim=1).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    return total_loss / total, correct / total, auc


# ─── TRAIN ────────────────────────────────────────────────────────────────────
model     = PRCCNv2(CONFIG).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val_auc = 0.0

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc      = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d}/{CONFIG['epochs']}  "
          f"train_loss: {train_loss:.4f}  train_acc: {train_acc:.4f}  "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}  val_auc: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), "prccnv2_kmnist_best.pth")
        print(f"  >> best model saved (val_auc: {val_auc:.4f})")

print(f"\nTraining complete. Best val_auc: {best_val_auc:.4f}")

PRCCNv2 ready
  patch tensor  : (B, 16, 2, 128)
  flat_size     : 64
  total params  : 114,186
  -- row kernels: 2,064
  -- col kernels: 2,064
  -- conv2d     : 59,232
  -- fc         : 50,826
Epoch 01/15  train_loss: 0.6654  train_acc: 0.7841  val_loss: 0.6951  val_acc: 0.7722  val_auc: 0.9727
  >> best model saved (val_auc: 0.9727)
Epoch 02/15  train_loss: 0.3302  train_acc: 0.8982  val_loss: 0.5416  val_acc: 0.8257  val_auc: 0.9829
  >> best model saved (val_auc: 0.9829)
Epoch 03/15  train_loss: 0.2563  train_acc: 0.9212  val_loss: 0.4492  val_acc: 0.8652  val_auc: 0.9871
  >> best model saved (val_auc: 0.9871)
Epoch 04/15  train_loss: 0.2172  train_acc: 0.9339  val_loss: 0.4416  val_acc: 0.8622  val_auc: 0.9880
  >> best model saved (val_auc: 0.9880)
Epoch 05/15  train_loss: 0.1906  train_acc: 0.9410  val_loss: 0.3844  val_acc: 0.8793  val_auc: 0.9914
  >> best model saved (val_auc: 0.9914)
Epoch 06/15  train_loss: 0.1741  train_acc: 0.9468  val_loss: 0.4027  val_acc: 0.8734  val_a

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

# ─── DATASET DOWNLOAD ─────────────────────────────────────────────────────────
# run this cell only once or when switching dataset
# change dataset name here only

DATA_DIR  = "./kmnist_data"
DOWNLOAD  = True

# download only — no transform yet, transform applied in cell 2
raw_train = datasets.KMNIST(DATA_DIR, train=True,  download=DOWNLOAD)
raw_val   = datasets.KMNIST(DATA_DIR, train=False, download=DOWNLOAD)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device  : {device}")
print(f"Train samples : {len(raw_train)}")
print(f"Val samples   : {len(raw_val)}")
print("Dataset ready — run cell 2 to configure and train")
# ──────────────────────────────────────────────────────────────────────────────

Using device  : cuda
Train samples : 60000
Val samples   : 10000
Dataset ready — run cell 2 to configure and train


In [4]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset

# ─── CV CONFIG ────────────────────────────────────────────────────────────────
CV_CONFIG = {
    "image_size"    : 128,
    "in_channels"   : 1,
    "num_classes"   : 10,
    "conv2d_layers" : [
        (32,  2, 3, 1, 1),
        (64,  1, 3, 1, 2),
        (128, 1, 3, 1, 2),
        (64,  1, 3, 1, 2),
    ],
    "fc_layers"     : [128, 64],
    "dropout"       : 0.1,
    "batch_size"    : 64,
    "epochs"        : 20,
    "lr"            : 1e-3,
    "n_folds"       : 5,
}
# ──────────────────────────────────────────────────────────────────────────────


class RowExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size  = cfg["image_size"]
        in_ch     = cfg["in_channels"]
        self.conv = nn.Conv2d(in_ch, 16,
                              kernel_size=(1, img_size),
                              stride=1, padding=0)
        self.img_size = img_size

    def forward(self, x):
        o = self.conv(x).squeeze(-1)
        o = F.adaptive_avg_pool1d(o, self.img_size)
        return o   # (B, 16, 128)


class ColExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        img_size  = cfg["image_size"]
        in_ch     = cfg["in_channels"]
        self.conv = nn.Conv2d(in_ch, 16,
                              kernel_size=(img_size, 1),
                              stride=1, padding=0)
        self.img_size = img_size

    def forward(self, x):
        o = self.conv(x).squeeze(2)
        o = F.adaptive_avg_pool1d(o, self.img_size)
        return o   # (B, 16, 128)


class PRCCNv2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.row_extractor = RowExtractor(cfg)
        self.col_extractor = ColExtractor(cfg)

        conv2d_blocks = []
        in_ch = 16
        for (out_ch, kh, kw, sh, sw) in cfg["conv2d_layers"]:
            conv2d_blocks += [
                nn.Conv2d(in_ch, out_ch,
                          kernel_size=(kh, kw),
                          stride=(sh, sw),
                          padding=(0, kw // 2)),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
            ]
            in_ch = out_ch
        self.conv2d_blocks = nn.Sequential(*conv2d_blocks)
        self.global_pool   = nn.AdaptiveAvgPool2d((1, 1))
        flat_size          = in_ch   # 64

        fc_blocks = []
        prev = flat_size
        for hidden in cfg["fc_layers"]:
            fc_blocks += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(cfg["dropout"])]
            prev = hidden
        fc_blocks.append(nn.Linear(prev, cfg["num_classes"]))
        self.fc = nn.Sequential(*fc_blocks)

    def forward(self, x):
        row_out = self.row_extractor(x).unsqueeze(2)   # (B, 16, 1, 128)
        col_out = self.col_extractor(x).unsqueeze(2)   # (B, 16, 1, 128)
        out     = torch.cat([row_out, col_out], dim=2) # (B, 16, 2, 128)
        out     = self.conv2d_blocks(out)              # (B, 64, 1, 16)
        out     = self.global_pool(out).flatten(1)     # (B, 64)
        return self.fc(out)


def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.squeeze().long().to(device)
            if train:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            all_probs.append(torch.softmax(out, dim=1).cpu().detach().numpy())
            all_labels.append(labels.cpu().detach().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    return total_loss / total, correct / total, auc


# ─── 5-FOLD CV ────────────────────────────────────────────────────────────────
# combine train and val into full dataset for CV
full_dataset = torch.utils.data.ConcatDataset([train_ds, val_ds])
all_labels   = np.array(
    [train_ds.targets[i] for i in range(len(train_ds))] +
    [val_ds.targets[i]   for i in range(len(val_ds))]
)

skf = StratifiedKFold(n_splits=CV_CONFIG["n_folds"], shuffle=True, random_state=42)

fold_results = []
total_params = None

print("="*65)
print("PRCCNv2 — 5-Fold Cross Validation — KMNIST")
print("="*65)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_labels)), all_labels)):
    print(f"\n── Fold {fold+1}/{CV_CONFIG['n_folds']} ──────────────────────────────────")

    fold_train = Subset(full_dataset, train_idx)
    fold_val   = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(fold_train, batch_size=CV_CONFIG["batch_size"], shuffle=True,  num_workers=2)
    fold_val_loader   = DataLoader(fold_val,   batch_size=CV_CONFIG["batch_size"], shuffle=False, num_workers=2)

    model     = PRCCNv2(CV_CONFIG).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CV_CONFIG["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    if total_params is None:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total params: {total_params:,}")

    best_fold_auc = 0.0

    for epoch in range(1, CV_CONFIG["epochs"] + 1):
        tr_loss, tr_acc, tr_auc = run_epoch(model, fold_train_loader, criterion, optimizer, device, train=True)
        vl_loss, vl_acc, vl_auc = run_epoch(model, fold_val_loader,   criterion, None,      device, train=False)
        scheduler.step(vl_loss)

        print(f"  Epoch {epoch:02d}/{CV_CONFIG['epochs']}  "
              f"tr_loss: {tr_loss:.4f}  tr_acc: {tr_acc:.4f}  "
              f"vl_loss: {vl_loss:.4f}  vl_acc: {vl_acc:.4f}  vl_auc: {vl_auc:.4f}")

        if vl_auc > best_fold_auc:
            best_fold_auc = vl_auc

    fold_results.append(best_fold_auc)
    print(f"  Fold {fold+1} best AUC: {best_fold_auc:.4f}")

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
fold_results = np.array(fold_results)
print("\n" + "="*65)
print("PRCCNv2 CV SUMMARY")
print("="*65)
print(f"Fold AUCs : {[f'{x:.4f}' for x in fold_results]}")
print(f"Mean AUC  : {fold_results.mean():.4f}")
print(f"Std AUC   : {fold_results.std():.4f}")
print(f"Params    : {total_params:,}")

PRCCNv2 — 5-Fold Cross Validation — KMNIST

── Fold 1/5 ──────────────────────────────────
Total params: 80,586
  Epoch 01/20  tr_loss: 0.7162  tr_acc: 0.7664  vl_loss: 0.3858  vl_acc: 0.8776  vl_auc: 0.9915
  Epoch 02/20  tr_loss: 0.3562  tr_acc: 0.8887  vl_loss: 0.2700  vl_acc: 0.9138  vl_auc: 0.9953
  Epoch 03/20  tr_loss: 0.2700  tr_acc: 0.9153  vl_loss: 0.2354  vl_acc: 0.9252  vl_auc: 0.9964
  Epoch 04/20  tr_loss: 0.2252  tr_acc: 0.9296  vl_loss: 0.2078  vl_acc: 0.9327  vl_auc: 0.9972
  Epoch 05/20  tr_loss: 0.1972  tr_acc: 0.9380  vl_loss: 0.1731  vl_acc: 0.9455  vl_auc: 0.9978
  Epoch 06/20  tr_loss: 0.1765  tr_acc: 0.9446  vl_loss: 0.2439  vl_acc: 0.9238  vl_auc: 0.9966
  Epoch 07/20  tr_loss: 0.1609  tr_acc: 0.9501  vl_loss: 0.1580  vl_acc: 0.9505  vl_auc: 0.9982
  Epoch 08/20  tr_loss: 0.1482  tr_acc: 0.9535  vl_loss: 0.1785  vl_acc: 0.9451  vl_auc: 0.9979
  Epoch 09/20  tr_loss: 0.1374  tr_acc: 0.9563  vl_loss: 0.1549  vl_acc: 0.9527  vl_auc: 0.9982
  Epoch 10/20  tr_loss: 

In [5]:
# ─── CNN CV CONFIG ────────────────────────────────────────────────────────────
CNN_CV_CONFIG = {
    "in_channels" : 1,
    "num_classes" : 10,
    "fc_layers"   : [128, 64],
    "dropout"     : 0.1,
    "batch_size"  : 64,
    "epochs"      : 20,
    "lr"          : 1e-3,
    "n_folds"     : 5,
}


class ComparableCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(cfg["in_channels"], 32,  kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),  nn.ReLU(),

            nn.Conv2d(32,  64,  kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),  nn.ReLU(),

            nn.Conv2d(64,  128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 64,  kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),  nn.ReLU(),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        flat_size        = 64

        fc_blocks = []
        prev = flat_size
        for hidden in cfg["fc_layers"]:
            fc_blocks += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(cfg["dropout"])]
            prev = hidden
        fc_blocks.append(nn.Linear(prev, cfg["num_classes"]))
        self.fc = nn.Sequential(*fc_blocks)

    def forward(self, x):
        out = self.global_pool(self.features(x))
        return self.fc(out.flatten(1))


# reuse run_epoch from cell 1
# reuse full_dataset and all_labels from cell 1

cnn_fold_results = []
cnn_total_params = None

print("="*65)
print("CNN — 5-Fold Cross Validation — KMNIST")
print("="*65)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_labels)), all_labels)):
    print(f"\n── Fold {fold+1}/{CNN_CV_CONFIG['n_folds']} ──────────────────────────────────")

    fold_train = Subset(full_dataset, train_idx)
    fold_val   = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(fold_train, batch_size=CNN_CV_CONFIG["batch_size"], shuffle=True,  num_workers=2)
    fold_val_loader   = DataLoader(fold_val,   batch_size=CNN_CV_CONFIG["batch_size"], shuffle=False, num_workers=2)

    model     = ComparableCNN(CNN_CV_CONFIG).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CNN_CV_CONFIG["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    if cnn_total_params is None:
        cnn_total_params = sum(p.numel() for p in model.parameters())
        print(f"Total params: {cnn_total_params:,}")

    best_fold_auc = 0.0

    for epoch in range(1, CNN_CV_CONFIG["epochs"] + 1):
        tr_loss, tr_acc, tr_auc = run_epoch(model, fold_train_loader, criterion, optimizer, device, train=True)
        vl_loss, vl_acc, vl_auc = run_epoch(model, fold_val_loader,   criterion, None,      device, train=False)
        scheduler.step(vl_loss)

        print(f"  Epoch {epoch:02d}/{CNN_CV_CONFIG['epochs']}  "
              f"tr_loss: {tr_loss:.4f}  tr_acc: {tr_acc:.4f}  "
              f"vl_loss: {vl_loss:.4f}  vl_acc: {vl_acc:.4f}  vl_auc: {vl_auc:.4f}")

        if vl_auc > best_fold_auc:
            best_fold_auc = vl_auc

    cnn_fold_results.append(best_fold_auc)
    print(f"  Fold {fold+1} best AUC: {best_fold_auc:.4f}")

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
cnn_fold_results = np.array(cnn_fold_results)
print("\n" + "="*65)
print("CNN CV SUMMARY")
print("="*65)
print(f"Fold AUCs : {[f'{x:.4f}' for x in cnn_fold_results]}")
print(f"Mean AUC  : {cnn_fold_results.mean():.4f}")
print(f"Std AUC   : {cnn_fold_results.std():.4f}")
print(f"Params    : {cnn_total_params:,}")

# ─── FINAL COMPARISON ─────────────────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL CV COMPARISON — KMNIST")
print("="*65)
print(f"{'Model':<12} {'Params':>10} {'Mean AUC':>10} {'Std AUC':>10}")
print("-"*45)
print(f"{'PRCCNv2':<12} {total_params:>10,} {fold_results.mean():>10.4f} {fold_results.std():>10.4f}")
print(f"{'CNN':<12} {cnn_total_params:>10,} {cnn_fold_results.mean():>10.4f} {cnn_fold_results.std():>10.4f}")

CNN — 5-Fold Cross Validation — KMNIST

── Fold 1/5 ──────────────────────────────────
Total params: 184,266
  Epoch 01/20  tr_loss: 1.1316  tr_acc: 0.6114  vl_loss: 0.6114  vl_acc: 0.8002  vl_auc: 0.9774
  Epoch 02/20  tr_loss: 0.5747  tr_acc: 0.8151  vl_loss: 0.4823  vl_acc: 0.8478  vl_auc: 0.9881
  Epoch 03/20  tr_loss: 0.4121  tr_acc: 0.8702  vl_loss: 0.3802  vl_acc: 0.8821  vl_auc: 0.9926
  Epoch 04/20  tr_loss: 0.3193  tr_acc: 0.8984  vl_loss: 0.3307  vl_acc: 0.8976  vl_auc: 0.9942
  Epoch 05/20  tr_loss: 0.2652  tr_acc: 0.9169  vl_loss: 0.2910  vl_acc: 0.9094  vl_auc: 0.9951
  Epoch 06/20  tr_loss: 0.2202  tr_acc: 0.9306  vl_loss: 0.3279  vl_acc: 0.8966  vl_auc: 0.9950
  Epoch 07/20  tr_loss: 0.1930  tr_acc: 0.9380  vl_loss: 0.1944  vl_acc: 0.9398  vl_auc: 0.9974
  Epoch 08/20  tr_loss: 0.1729  tr_acc: 0.9450  vl_loss: 0.2896  vl_acc: 0.9118  vl_auc: 0.9961
  Epoch 09/20  tr_loss: 0.1521  tr_acc: 0.9521  vl_loss: 0.2378  vl_acc: 0.9241  vl_auc: 0.9966
  Epoch 10/20  tr_loss: 0.1